In [18]:
# ODIN v11: THE STRATEGIC GROWTH ORACLE (Read-Only Edition)
# Focus: Quality of Earnings, Revenue Concentration, and Operational Integrity

import os
import io
import xmlrpc.client
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from dotenv import load_dotenv

# PDF Generation (ReportLab)
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image as RLImage
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch

# ================================================================
# 1. READ-ONLY ODOO CONNECTOR (SAFETY FIRST)
# ================================================================
load_dotenv()
plt.switch_backend('Agg')

class OdooOracle:
    def __init__(self):
        self.url = os.getenv('ODOO_URL', 'https://vendyltd.odoo.com/')
        self.db = os.getenv('ODOO_DB', 'vendyltd')
        self.user = os.getenv('ODOO_USER', 'muktadir@vendy.ltd')
        self.pwd = os.getenv('ODOO_PASSWORD', '205958c9bb841b4bef7e6da4a3afb5b5029cd6ae')
        
        common = xmlrpc.client.ServerProxy(f"{self.url}/xmlrpc/2/common")
        self.uid = common.authenticate(self.db, self.user, self.pwd, {})
        self.models = xmlrpc.client.ServerProxy(f"{self.url}/xmlrpc/2/object")
        print(f"✅ Oracle Authenticated (UID: {self.uid})")

    def read(self, model, domain, fields):
        """Strict search_read only to prevent database modification."""
        return self.models.execute_kw(self.db, self.uid, self.pwd, model, 'search_read', [domain], {'fields': fields})

# ================================================================
# 2. STRATEGIC INTELLIGENCE ENGINE
# ================================================================
class IntelligenceEngine:
    @staticmethod
    def get_survival_metrics(oracle):
        """Analyzes Quality of Earnings and Debtor Concentration."""
        moves = oracle.read('account.move', [('state', '=', 'posted')], 
                           ['amount_total', 'move_type', 'payment_state', 'partner_id', 'amount_residual'])
        df = pd.DataFrame(moves)
        
        profit = df[df['move_type'] == 'out_invoice']['amount_total'].sum()
        cash = df[(df['move_type'] == 'out_invoice') & (df['payment_state'] == 'paid')]['amount_total'].sum()
        
        # Debtor Concentration Drill-Down [cite: 223]
        unpaid = df[df['payment_state'] != 'paid'].copy()
        unpaid['partner_name'] = unpaid['partner_id'].apply(lambda x: x[1] if isinstance(x, list) else 'Unknown')
        concentration = unpaid.groupby('partner_name')['amount_residual'].sum().sort_values(ascending=False).to_dict()
        
        return {
            "profit": profit, 
            "cash": cash, 
            "qoe": (cash / profit) if profit > 0 else 0,
            "concentration": concentration
        }

    @staticmethod
    def get_operational_risks(oracle):
        """Audits Ghost Assets and Fulfillment Bottlenecks[cite: 234]."""
        ghosts = oracle.read('stock.quant', [('quantity', '<', 0)], ['product_id', 'quantity'])
        pickings = oracle.read('stock.picking', [], ['state'])
        df_p = pd.DataFrame(pickings)
        completion_rate = (len(df_p[df_p['state'] == 'done']) / len(df_p)) * 100 if not df_p.empty else 0
        
        return {"ghost_count": len(ghosts), "completion_rate": round(completion_rate, 1)}

# ================================================================
# 3. EXECUTIVE REPORTER (VISUAL STRATEGY)
# ================================================================
class ExecutiveReporter:
    def __init__(self):
        self.styles = getSampleStyleSheet()
        self.color_navy = colors.HexColor('#1a237e')

    def plot_concentration(self, concentration_dict):
        """Generates the Revenue Concentration Risk Chart[cite: 223]."""
        plt.figure(figsize=(5, 3))
        top_names = [n[:15] for n in list(concentration_dict.keys())[:3]] # Truncate for display
        top_values = list(concentration_dict.values())[:3]
        plt.bar(top_names, top_values, color=['#c62828', '#e67e22', '#f9a825'])
        plt.title("REVENUE CONCENTRATION RISK (Top 3 Debtors)", fontweight='bold')
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=150)
        buf.seek(0)
        return buf

    def generate_pdf(self, metrics, risks, filename="CEO_GROWTH_ORACLE_V11.pdf"):
        doc = SimpleDocTemplate(filename, pagesize=letter)
        story = []
        
        # Header
        story.append(Paragraph("ODIN STRATEGIC GROWTH ORACLE v11", self.styles['Title']))
        story.append(Spacer(1, 20))
        
        # Financial Section
        story.append(Paragraph("1. FINANCIAL SURVIVAL STATUS", self.styles['Heading2']))
        story.append(RLImage(self.plot_concentration(metrics['concentration']), width=4*inch, height=2.5*inch))
        
        top_debtor = list(metrics['concentration'].keys())[0]
        top_amt = list(metrics['concentration'].values())[0]
        
        story.append(Paragraph(f"<b>Top Debtor Risk:</b> {top_debtor} owes ${top_amt:,.2f}[cite: 233].", self.styles['Normal']))
        story.append(Paragraph(f"<b>Quality of Earnings:</b> {metrics['qoe']:.2f}[cite: 216, 233].", self.styles['Normal']))
        story.append(Spacer(1, 20))
        
        # Operational Section
        story.append(Paragraph("2. OPERATIONAL BOTTLENECKS", self.styles['Heading2']))
        story.append(Paragraph(f"• <b>Completion Rate:</b> {risks['completion_rate']}%[cite: 235].", self.styles['Normal']))
        story.append(Paragraph(f"• <b>Ghost Assets:</b> {risks['ghost_count']} units.", self.styles['Normal']))
        
        doc.build(story)
        print(f"✅ Strategic Oracle Report saved as: {filename}")

# ================================================================
# 4. MAIN ORCHESTRATOR
# ================================================================
if __name__ == "__main__":
    oracle = OdooOracle()
    intel = IntelligenceEngine()
    rep = ExecutiveReporter()
    
    # Analyze
    m = intel.get_survival_metrics(oracle)
    r = intel.get_operational_risks(oracle)
    
    # Report
    rep.generate_pdf(m, r)

✅ Oracle Authenticated (UID: 2)
✅ Strategic Oracle Report saved as: CEO_GROWTH_ORACLE_V11.pdf
